<div style="background: linear-gradient(135deg, #0a1628 0%, #1a3a5c 50%, #2d6a4f 100%); padding: 40px 30px; border-radius: 16px; margin-bottom: 24px; color: white; font-family: 'Segoe UI', sans-serif;">
  <h1 style="margin: 0 0 8px 0; font-size: 2em;">🔀 Lab 03b: Build a Multi-Agent Advisory Workflow</h1>
  <p style="margin: 0; font-size: 1.15em; opacity: 0.9;">Zava Bank Workshop — Microsoft Foundry Agent Observability</p>
</div>

## 📚 What We Learn

In this lab we orchestrate **multiple specialist agents** into a coordinated **workflow**. Each agent has its own area of expertise; the workflow coordinates them to produce a comprehensive client advisory.

| Concept | Detail |
| --- | --- |
| **Multi-agent orchestration** | A `WorkflowAgentDefinition` that chains specialist agents together |
| **Specialist agents** | Portfolio Analyst, Risk Assessor, Market Researcher |
| **Synthesis** | A Client Advisor agent that merges specialist outputs into one coherent summary |
| **Streaming events** | Real-time visibility into each workflow action as it executes |

## 🏦 The Zava Bank Story

A single advisor agent with tools works well for straightforward requests, but a **real advisory team** has specialists:

- **Portfolio Analyst** — focuses on holdings, allocation, performance, and gains/losses  
- **Risk Assessor** — evaluates exposure, beta, concentration, sector overweight, and compliance  
- **Market Researcher** — tracks indices, sector trends, economic indicators, and macro context  
- **Client Advisor** — orchestrates the specialists and synthesizes everything into a single, actionable advisory  

This mirrors how wealth management actually works: each desk has a specialty, and the relationship manager pulls it all together for the client.

---
## 1 · Setup

In [ ]:
import os
from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import (
    PromptAgentDefinition,
    WorkflowAgentDefinition,
)

load_dotenv(
    dotenv_path=os.path.join(os.path.dirname(os.path.abspath(".")), ".env")
)

endpoint = os.environ["AZURE_AI_PROJECT_ENDPOINT"]
model_name = os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"]
credential = DefaultAzureCredential()

project_client = AIProjectClient(
    endpoint=endpoint,
    credential=credential,
    allow_preview=True,   # required for WorkflowAgentDefinition
)

print(f"Project endpoint : {endpoint}")
print(f"Model deployment : {model_name}")

---
## 2 · Create Specialist Agents

### 2a — Portfolio Analyst

In [ ]:
portfolio_agent = project_client.agents.create_version(
    agent_name="zava-portfolio-analyst",
    definition=PromptAgentDefinition(
        model=model_name,
        instructions=(
            "You are a Portfolio Analyst at Zava Bank.\n\n"
            "Your job:\n"
            "- Analyze client holdings and current allocation.\n"
            "- Calculate realized and unrealized gains/losses.\n"
            "- Identify concentration risk in individual positions.\n"
            "- Note any drift from the target allocation.\n\n"
            "Formatting rules:\n"
            "- Use bullet points for clarity.\n"
            "- Include percentages and dollar amounts where relevant.\n"
            "- End your response with: [PORTFOLIO ANALYSIS COMPLETE]"
        ),
    ),
)

print(f"✅ Portfolio Analyst  — name={portfolio_agent.name}  version={portfolio_agent.version}")

### 2b — Risk Assessor

In [ ]:
risk_agent = project_client.agents.create_version(
    agent_name="zava-risk-assessor",
    definition=PromptAgentDefinition(
        model=model_name,
        instructions=(
            "You are a Risk Assessor at Zava Bank.\n\n"
            "Your job:\n"
            "- Evaluate portfolio risk metrics (beta, volatility, drawdown).\n"
            "- Flag concerns: high-beta positions, concentration > 15%% in one name, "
            "sector overweight vs. benchmark.\n"
            "- Check for compliance with the client's stated risk tolerance.\n"
            "- Suggest mitigation strategies (hedging, rebalancing, diversification).\n\n"
            "Formatting rules:\n"
            "- Organize by risk category.\n"
            "- Use a severity indicator (🟢 Low / 🟡 Medium / 🔴 High) for each finding.\n"
            "- End your response with: [RISK ASSESSMENT COMPLETE]"
        ),
    ),
)

print(f"✅ Risk Assessor     — name={risk_agent.name}  version={risk_agent.version}")

### 2c — Market Researcher

In [ ]:
market_agent = project_client.agents.create_version(
    agent_name="zava-market-researcher",
    definition=PromptAgentDefinition(
        model=model_name,
        instructions=(
            "You are a Market Researcher at Zava Bank.\n\n"
            "Your job:\n"
            "- Provide relevant market context: major index performance, sector trends, "
            "and key economic indicators.\n"
            "- Connect macro data points to the client's portfolio implications.\n"
            "- Highlight upcoming catalysts (earnings, Fed meetings, geopolitical events).\n\n"
            "Formatting rules:\n"
            "- Organize by theme (macro, sector, catalysts).\n"
            "- Include directional indicators (↑ / ↓ / →).\n"
            "- End your response with: [MARKET RESEARCH COMPLETE]"
        ),
    ),
)

print(f"✅ Market Researcher — name={market_agent.name}  version={market_agent.version}")

### 2d — Client Advisor (synthesis / concierge)

In [ ]:
advisor_agent = project_client.agents.create_version(
    agent_name="zava-client-advisor",
    definition=PromptAgentDefinition(
        model=model_name,
        instructions=(
            "You are the Client Advisor at Zava Bank — the relationship manager who "
            "synthesizes specialist analysis into one cohesive advisory.\n\n"
            "Your job:\n"
            "- Combine inputs from the Portfolio Analyst, Risk Assessor, and Market Researcher.\n"
            "- Produce a clear, actionable advisory summary.\n"
            "- Include specific recommendations (buy/sell/hold/rebalance) with rationale.\n"
            "- Add standard compliance disclaimers.\n\n"
            "Formatting rules:\n"
            "- Use numbered sections: Executive Summary, Key Findings, Recommendations, Disclaimers.\n"
            "- Write for a human financial advisor to review before forwarding to the client.\n"
            "- Keep the tone professional and concise."
        ),
    ),
)

print(f"✅ Client Advisor    — name={advisor_agent.name}  version={advisor_agent.version}")

---
## 3 · Define the Workflow (YAML)

In [ ]:
workflow_yaml = f"""
kind: workflow
trigger:
  kind: OnConversationStart
  id: zava_advisory_workflow
  actions:
    # ── Capture the incoming request ────────────────────────
    - kind: SetVariable
      id: capture_advisor_request
      variable: Local.LatestMessage
      value: "=UserMessage(System.LastMessageText)"

    # ── Create separate conversations for each specialist ──
    - kind: CreateConversation
      id: create_portfolio_conversation
      conversationId: Local.PortfolioConversationId

    - kind: CreateConversation
      id: create_risk_conversation
      conversationId: Local.RiskConversationId

    - kind: CreateConversation
      id: create_market_conversation
      conversationId: Local.MarketConversationId

    # ── Invoke specialist agents ──────────────────────────
    - kind: InvokeAzureAgent
      id: invoke_portfolio_analyst
      description: Analyze client portfolio
      conversationId: "=Local.PortfolioConversationId"
      agent:
        name: {portfolio_agent.name}
        version: \"{portfolio_agent.version}\"
      activity: "=Local.LatestMessage"

    - kind: InvokeAzureAgent
      id: invoke_risk_assessor
      description: Assess portfolio risk
      conversationId: "=Local.RiskConversationId"
      agent:
        name: {risk_agent.name}
        version: \"{risk_agent.version}\"
      activity: "=Local.LatestMessage"

    - kind: InvokeAzureAgent
      id: invoke_market_researcher
      description: Research market conditions
      conversationId: "=Local.MarketConversationId"
      agent:
        name: {market_agent.name}
        version: \"{market_agent.version}\"
      activity: "=Local.LatestMessage"

    # ── Synthesize all specialist outputs ─────────────────
    - kind: InvokeAzureAgent
      id: synthesize_with_advisor
      description: Synthesize into advisory summary
      conversationId: "=System.ConversationId"
      agent:
        name: {advisor_agent.name}
        version: \"{advisor_agent.version}\"
      activity: "Based on the specialist analysis, provide a comprehensive advisory summary for the following request: =Local.LatestMessage"
"""

print("Workflow YAML defined ✅")
print(f"Specialist agents referenced:")
print(f"  📊 {portfolio_agent.name} v{portfolio_agent.version}")
print(f"  ⚠️  {risk_agent.name} v{risk_agent.version}")
print(f"  📈 {market_agent.name} v{market_agent.version}")
print(f"  🏦 {advisor_agent.name} v{advisor_agent.version}")

---
## 4 · Create the Workflow Agent

In [ ]:
workflow = project_client.agents.create_version(
    agent_name="zava-advisory-workflow",
    definition=WorkflowAgentDefinition(workflow=workflow_yaml),
)

print(f"✅ Workflow agent created — name={workflow.name}  version={workflow.version}")

---
## 5 · Run the Advisory Workflow

In [ ]:
from azure.ai.projects.models import (
    MessageRole,
    AgentStreamEvent,
)

test_prompt = (
    "Prepare a comprehensive quarterly review for client Apex Capital Partners. "
    "I need portfolio analysis, risk assessment, and market context."
)

print(f"🗣️  Prompt: {test_prompt}\n")
print("=" * 70)

# Track conversation IDs created by the workflow
sub_conversation_ids: dict[str, str] = {}
final_response = ""

with project_client.agents.run(
    agent_name=workflow.name,
    agent_version=str(workflow.version),
    messages=[
        {"role": MessageRole.USER, "content": test_prompt}
    ],
    stream=True,
) as stream:
    for event in stream:
        kind = getattr(event, "kind", None) or getattr(event, "type", None) or ""

        # Workflow action lifecycle events
        if kind == "workflow_action":
            action_data = getattr(event, "data", event)
            action_id = getattr(action_data, "action_id", "")
            status = getattr(action_data, "status", "")

            if status == "started":
                print(f"  ▶️  Action started : {action_id}")
            elif status == "completed":
                print(f"  ✅ Action complete: {action_id}")

                # Capture sub-conversation IDs when conversations are created
                conv_id = getattr(action_data, "conversation_id", None)
                if conv_id and action_id.startswith("create_"):
                    label = action_id.replace("create_", "").replace("_conversation", "")
                    sub_conversation_ids[label] = conv_id

        # Streaming text chunks from the final synthesis
        elif kind in ("content.delta", "message.delta"):
            delta = getattr(event, "data", event)
            text = getattr(delta, "content", "") or getattr(delta, "text", "")
            if text:
                final_response += text
                print(text, end="", flush=True)

print("\n" + "=" * 70)
print("\n🏁 Workflow execution complete.")
print(f"Sub-conversations captured: {list(sub_conversation_ids.keys())}")

---
## 6 · Retrieve Specialist Responses

Each specialist ran in its own conversation. Let's pull their individual outputs.

In [ ]:
label_map = {
    "portfolio": "📊 Portfolio Analyst",
    "risk":      "⚠️  Risk Assessor",
    "market":    "📈 Market Researcher",
}

for key, conv_id in sub_conversation_ids.items():
    label = label_map.get(key, key)
    print(f"\n{'━' * 70}")
    print(f"{label}  (conversation: {conv_id})")
    print(f"{'━' * 70}")

    try:
        messages = project_client.agents.list_messages(conversation_id=conv_id)
        for msg in reversed(list(messages)):
            role = getattr(msg, "role", "unknown")
            content = getattr(msg, "content", "")
            # Content may be a list of content blocks
            if isinstance(content, list):
                text_parts = []
                for block in content:
                    text_val = getattr(block, "text", None)
                    if text_val:
                        text_parts.append(
                            getattr(text_val, "value", str(text_val))
                        )
                content = "\n".join(text_parts)
            print(f"\n[{role}]\n{content}")
    except Exception as e:
        print(f"  ⚠️  Could not retrieve messages: {e}")

# Final synthesis
print(f"\n{'━' * 70}")
print("🏦 Client Advisor — Advisory Summary")
print(f"{'━' * 70}")
print(final_response or "(captured in streaming output above)")

---
## 7 · Cleanup

In [ ]:
# Delete workflow agent
project_client.agents.delete_agent(agent_name=workflow.name)
print(f"🗑️  Deleted workflow : {workflow.name}")

# Delete specialist agents
for agent in [portfolio_agent, risk_agent, market_agent, advisor_agent]:
    project_client.agents.delete_agent(agent_name=agent.name)
    print(f"🗑️  Deleted agent    : {agent.name}")

# Delete sub-conversations
for key, conv_id in sub_conversation_ids.items():
    try:
        project_client.agents.delete_conversation(conversation_id=conv_id)
        print(f"🗑️  Deleted conversation: {key} ({conv_id})")
    except Exception:
        pass

# Close clients
project_client.close()

print("\n✅ All resources cleaned up.")

---
## ✅ Summary

<div style="background: #f0f7f4; border-left: 4px solid #2d6a4f; padding: 16px 20px; border-radius: 8px; margin-top: 12px;">

**What we accomplished:**

1. Created **three specialist agents** — each with focused expertise (portfolio, risk, market)  
2. Created a **Client Advisor agent** that synthesizes specialist outputs  
3. Defined a **WorkflowAgentDefinition** (YAML) that orchestrates all four agents  
4. Ran the full workflow with **streaming event visibility** into each action  
5. Retrieved individual specialist responses from their sub-conversations  

**Key takeaway:** Multi-agent workflows give you **separation of concerns** for complex advisory tasks. Each specialist can be tuned, evaluated, and swapped independently — while the workflow guarantees they all contribute to the final output.

**Next steps:** In the tracing labs, we'll add observability to see exactly how long each specialist takes and where latency hides in a multi-agent workflow.

</div>